## Imports

In [1]:
import json
import time
from openai import OpenAI
from google.colab import userdata, drive
import csv
import json
import argparse
import time
import sys
from pathlib import Path
from datetime import datetime
import pandas as pd
from abc import ABC, abstractmethod
import ast
from tqdm import tqdm
import os

In [2]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Extract Data

In [3]:
matching_df = pd.read_csv("/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/matching_report.csv")
matching_df

,nct_id,eligible_count,subject_ids
0,NCT03319472,85,"[10035787, 10093609, 10194132, 10233845, 10362..."
1,NCT00236236,467,"[10237315, 10427443, 11149909, 11156530, 11167..."
2,NCT01550107,269,"[10345771, 10352192, 10362783, 10363534, 10368..."
3,NCT00356044,173,"[10096420, 10227947, 10247940, 10250159, 10276..."
4,NCT02805387,0,[]
5,NCT01642784,82,"[10722545, 10779064, 10797890, 10827741, 10851..."
6,NCT01139307,0,[]
7,NCT02367716,244,"[10237315, 10250582, 10283092, 10488401, 10533..."
8,NCT00176384,0,[]
9,NCT01510652,198,"[10013569, 10014651, 10047893, 10067573, 10076..."


In [4]:
clinical_trials_df = pd.read_csv("/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/processed_studies.csv")
clinical_trials_df.head()

,nct_id,official_title,eligibility_criteria,inclusion_criteria,exclusion_criteria
0,NCT03319472,Clinical Identification of Malignant Pleural E...,Inclusion Criteria:\n\n* Pleural effusion\n* H...,"[""Pleural effusion"", ""Hospital admission"", ""No...","[""No diagnosis at one month post-admission"", ""..."
1,NCT00236236,CONTAK RENEWAL® Heart Failure Heart Rate Varia...,Inclusion Criteria:\n\n* Patients receiving th...,"[""Patients receiving their first CRT-D"", ""Pati...","[""Patients who are anticipated to receive paci..."
2,NCT01550107,A Prospective Study to Evaluate the Effect of ...,Inclusion Criteria:\n\nAge 65 and over 6-Minut...,"[""Age 65 and over"", ""6-Minute Walk Distance <4...","[""Documented history of peripheral arterial di..."
3,NCT00356044,Femoral Versus Radial Access for Coronary Inte...,Inclusion Criteria:\n\n* Patients with ST elev...,"[""Patients with ST elevation acute myocardial ...","[""Patients in cardiogenic shock were excluded ..."
4,NCT02805387,"Bicarbonate, a New Treatment of Labour Dystocia","Inclusion Criteria:\n\n* primiparity, singleto...","[""primiparity, singleton pregnancy"", ""with an ...","[""multiparous women"", ""deliveries with non-cep..."


## Transform

In [6]:
# @title BASE ABSTRACT CLASS
class BaseJudge(ABC):

    @abstractmethod
    def generate(self, system_prompt, user_prompt):
        pass

In [7]:
# @title GROQ JUDGE
class GroqJudge(BaseJudge):

    def __init__(self, api_key, model="deepseek-r1-distill-llama-70b"):
        self.client = OpenAI(
            api_key=api_key,
            # base_url="https://api.groq.com/openai/v1"
            base_url="https://openrouter.ai/api/v1"
        )
        self.model = model

    def generate(self, system_prompt, user_prompt):
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": user_prompt
                }
            ],
            temperature=0
        )

        return response.choices[0].message.content

In [8]:
# @title CLINICAL AUDITOR
class ClinicalMatchingAuditor:
    def __init__(self, judge, output_path="matching_audit_results.csv"):
        self.judge = judge
        self.output_path = output_path
        self.system_prompt = """
            You are a Senior Clinical Trial Coordinator and Data Auditor.
            Your task is to verify if the automated matching of a patient to a clinical trial is correct based on the eligibility criteria.

            EVALUATION RULES:
            1. Eligibility Assessment: Strictly verify if the patient satisfies the criteria.
            2. Principle of Absence (CRITICAL):
              - If a criterion checks for a "history of a condition" or "previous procedure" (e.g., 'previous pleural procedure', 'previous chemoradiotherapy'), and there is NO record of such event in the patient's data, you MUST assume the patient DID NOT have it.
              - In clinical screenings, the absence of a record for a specific medical event implies that the event did not occur. Therefore, the patient is considered to have "No previous history" of that event.
              - Do not flag this as a missing data issue or a failure to meet criteria. Instead, treat it as confirmation that the condition is absent.
            3. Accuracy: Identify logical gaps in the automated matching.
            4. Consent: Assume all patients have provided informed consent.
            5. Regarding temporal criteria (e.g., 'within 4 hours of admission'): If the clinical data (lab results, etc.) is present in the patient record, assume it was obtained within the appropriate timeframe for clinical evaluation unless there is explicit evidence to the contrary. Do not invalidate the matching based solely on missing timestamp documentation.

            Respond ONLY with valid JSON.
            """

    def _generate_user_prompt(self, inclusion_criteria, exclusion_criteria, patient_data, system_decision):
        return f"""
          STUDY PROTOCOL INCLUSION CRITERIA:
          {inclusion_criteria}

          STUDY PROTOCOL EXCLUSION CRITERIA:
          {exclusion_criteria}

          PATIENT CLINICAL DATA:
          {patient_data}

          SYSTEM DECISION:
          {system_decision}

          AUDIT TASK:
          Verify if the decision "Included" or "Excluded" is clinically consistent with the inclusion and exclusion criteria.

          Return ONLY valid JSON:
          {{
            "is_correct": boolean,
            "error_type": "None | False Positive (Included ineligible) | False Negative (Excluded eligible) | Missing Clinical Data",
            "justification": "Evidence-based reasoning citing specific criteria",
            "confidence_score": integer (1-5)
          }}
        """

    def audit_matching_case(self, nct_id, subject_id, inclusion_criteria, exclusion_criteria, patient_data, decision):
        # Aqui você monta o texto da decisão
        system_decision = f"Patient {subject_id} was {decision} in study {nct_id}"

        try:
            response_text = self.judge.generate(
                system_prompt=self.system_prompt,
                user_prompt=self._generate_user_prompt(inclusion_criteria, exclusion_criteria, patient_data, system_decision)
            )
            # Limpeza padrão
            response_text = response_text.replace("```json", "").replace("```", "").strip()
            parsed = json.loads(response_text)
            parsed.update({"nct_id": nct_id, "subject_id": subject_id})

            # --- SALVAMENTO INCREMENTAL AQUI ---
            df_result = pd.DataFrame([parsed])
            df_result.to_csv(
                self.output_path,
                mode='a',
                header=not os.path.exists(self.output_path),
                index=False
            )
            return parsed
        except Exception as e:
            print(f"Error auditing {nct_id}: {e}")
            return {"nct_id": nct_id, "subject_id": subject_id, "error": str(e)}


In [9]:
def prepare_audit_dataset(df_matching, df_patients, df_criteria):
    # 1. Merge dos critérios nos resultados de matching
    df = df_matching.merge(df_criteria[['nct_id', 'eligibility_criteria', 'inclusion_criteria', 'exclusion_criteria']], on='nct_id', how='left')

    # Convert string representation of lists to actual lists
    df['subject_ids'] = df['subject_ids'].apply(lambda s: ast.literal_eval(s) if isinstance(s, str) and s.startswith('[') else s)

    # 2. Explode the list of subject_ids
    df = df.explode('subject_ids')

    df_patients['subject_data'] = df_patients.apply(
      lambda row: {k: v for k, v in row.items() if pd.notna(v) and k != 'subject_id'}, axis=1
    )
    df_patients = df_patients[['subject_id', 'subject_data']]

    # Convert subject_ids to numeric type for merging
    df['subject_ids'] = pd.to_numeric(df['subject_ids'], errors='coerce')
    df.dropna(subset=['subject_ids'], inplace=True)
    df['subject_ids'] = df['subject_ids'].astype(int)

    # Merge com os dados detalhados do paciente
    final_df = df.merge(df_patients, left_on='subject_ids', right_on='subject_id', how='left')

    return final_df

In [10]:
# @title Subsets data
all_cols = mimic_patients_df.columns.tolist()

cols_obrigatorias = ['subject_id', 'gender', 'age', 'name_diag_principal', 'list_procedures', 'HPI', 'chief_complaint']
to_ignore = ['hadm_id']
cols_com_dados = mimic_patients_df.columns[mimic_patients_df.notna().sum() > 0].tolist()
filtered_cols = [c for c in cols_com_dados if c not in to_ignore and not c.endswith('_FLAG')]
cols_finais = list(set(cols_obrigatorias + filtered_cols))
patients_subset = mimic_patients_df[cols_finais].copy()

def trim_clinical_text(text, limit=600):
    text = str(text)
    if len(text) <= limit:
        return text

    # Pega os primeiros 300 e os últimos 300 caracteres
    half = limit // 2
    return text[:half] + "\n[...] [TEXT CUT] [...]\n" + text[-half:]

# Aplica no seu DataFrame
patients_subset['HPI'] = patients_subset['HPI'].apply(trim_clinical_text)

In [11]:
# full_df[(full_df['nct_id'] == 'NCT03319472') & (full_df['subject_id'] == 15613540)]

In [12]:
full_df = prepare_audit_dataset(matching_df, patients_subset, clinical_trials_df)
# full_df
judge = GroqJudge(
      # api_key=userdata.get("JudgeGROQAPI"),
      api_key=userdata.get("JudgeOpenRouter"),
      model="openai/gpt-oss-120b"
      # model="llama-3.3-70b-versatile"
)
auditor = ClinicalMatchingAuditor(
    judge=judge,
    output_path="/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/matching_audit_results.csv"
)
# for idx, row in full_df[(full_df['nct_id'] == 'NCT03319472') & (full_df['subject_id'] == 15613540)].iterrows():
for idx, row in full_df.iterrows():
    auditor.audit_matching_case(
        row['nct_id'], row['subject_ids'],
        row['inclusion_criteria'], row['exclusion_criteria'], row['subject_data'],
        "Included" # ou "Excluded" conforme seu match
    )

Error auditing NCT00236236: Expecting ',' delimiter: line 5 column 24 (char 739)
Error auditing NCT00236236: Unterminated string starting at: line 4 column 20 (char 100)
Error auditing NCT00236236: 'NoneType' object has no attribute 'replace'
Error auditing NCT00236236: Expecting ',' delimiter: line 5 column 24 (char 726)
Error auditing NCT00236236: Expecting ',' delimiter: line 5 column 24 (char 797)
Error auditing NCT01550107: Expecting ',' delimiter: line 5 column 24 (char 572)
Error auditing NCT01510652: Expecting ',' delimiter: line 5 column 24 (char 629)
Error auditing NCT01510652: Expecting ',' delimiter: line 5 column 24 (char 962)


In [13]:
# full_df.head()

In [14]:
df = pd.read_csv("/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/matching_audit_results.csv")
# df = df.head(0)
df
# df.to_csv("/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/matching_audit_results.csv", index=False)

,is_correct,error_type,justification,confidence_score,nct_id,subject_id
0,False,False Positive (Included ineligible),The patient has a documented pleural procedure...,4,NCT03319472,10035787
1,False,Missing Clinical Data,The patient meets all inclusion criteria (age ...,3,NCT03319472,10093609
2,True,NaN,"The patient is >18 years old, has documented n...",5,NCT03319472,10194132
3,False,False Positive (Included ineligible),The patient meets age and pleural effusion cri...,5,NCT03319472,10233845
4,True,NaN,"The patient is >18 years old, has documented p...",5,NCT03319472,10362783
...,...,...,...,...,...,...
1671,False,False Positive (Included ineligible),The patient meets basic criteria such as age (...,4,NCT01510652,19924718
1672,False,False Positive (Included ineligible),The patient meets age and has no documented ex...,3,NCT01510652,19954261
1673,False,False Positive (Included ineligible),"The patient meets age, consent, and willingnes...",4,NCT01510652,19966568
1674,False,Missing Clinical Data,"The patient meets age, consent, and pregnancy ...",2,NCT01510652,19969139
